# Document Scanner — Colab GPU Loss Ablation Launcher

This notebook runs the 4-loss ablation suite on Google Colab T4 GPU (1000 samples/epoch, 20 epochs):
1. **MSE Loss** (`exp-001_enh_mse`)
2. **L1 Loss** (`exp-002_enh_l1`)
3. **L1 + MS-SSIM Loss** (`exp-003_enh_l1msssim`)
4. **L1 + MS-SSIM + Sobel Loss** (`exp-004_enh_l1msssim_sobel`)

At 1000 samples/epoch on T4 GPU with AMP, each epoch takes ~15 seconds, and all 4 experiments will finish in **~20 minutes total**!

### Step 0: Check GPU Availability

In [ ]:
import torch
print('PyTorch Version:', torch.__version__)
cuda_avail = torch.cuda.is_available()
print('CUDA Available:', cuda_avail)
if cuda_avail:
    print('GPU Device Name:', torch.cuda.get_device_name(0))
else:
    print('❌ CRITICAL: CUDA is NOT detected by PyTorch!')
    print('👉 Action required: In top menu click Runtime -> Restart session (or Runtime -> Disconnect and delete runtime), then run again.')

### Step 1: Mount Google Drive & Clone Repository

In [ ]:
# Mount Google Drive (to easily load data.zip and save final checkpoints)
from google.colab import drive
drive.mount('/content/drive')

import os
if not os.path.exists('DocEn'):
    !git clone https://github.com/HedieTahmouresi/DocEn.git
%cd DocEn
!git pull origin main

### Step 2: Extract Data Directory
Unzips `data.zip` (uploaded to Google Drive `/content/drive/MyDrive/data.zip` or directly to Colab).

In [ ]:
import os
if not os.path.exists('data/clean_scans'):
    if os.path.exists('/content/drive/MyDrive/data.zip'):
        print('Extracting data.zip from Google Drive...')
        !unzip -q /content/drive/MyDrive/data.zip -d .
    elif os.path.exists('/content/data.zip'):
        print('Extracting data.zip from Colab root...')
        !unzip -q /content/data.zip -d .
    else:
        print('WARNING: data.zip not found! Please upload data.zip to Google Drive or Colab file manager.')
else:
    print('Data directory already exists!')

### Step 3: Install Requirements

In [ ]:
!pip install -r requirements.txt

### Step 4: Run Full Loss Ablation Suite on GPU

In [ ]:
# Run experiment 1: MSE Loss
!python train.py --config configs/exp/exp-001_enh_mse.yaml --env colab_t4

In [ ]:
# Run experiment 2: L1 Loss
!python train.py --config configs/exp/exp-002_enh_l1.yaml --env colab_t4

In [ ]:
# Run experiment 3: L1 + MS-SSIM Loss
!python train.py --config configs/exp/exp-003_enh_l1msssim.yaml --env colab_t4

In [ ]:
# Run experiment 4: L1 + MS-SSIM + Sobel Loss
!python train.py --config configs/exp/exp-004_enh_l1msssim_sobel.yaml --env colab_t4

### Step 5: Evaluate Ablation Results & Generate Comparison Plots

In [ ]:
!python -m scripts.evaluate_ablation

### Step 6: Package & Download Results (Also Saves to Google Drive)

In [ ]:
from google.colab import files
!zip -r phase04_ablation_results.zip runs/ outputs/figures/
# Copy zip to Google Drive as backup
!cp phase04_ablation_results.zip /content/drive/MyDrive/
print('Saved phase04_ablation_results.zip to Google Drive!')
# Trigger browser download
files.download('phase04_ablation_results.zip')